In [2]:
from __future__ import annotations

import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

## Load the data

In [3]:
DATA_DIR = Path.cwd()
MOVIES_FILE = DATA_DIR / "tmdb_5000_movies.csv"
CREDITS_FILE = DATA_DIR / "tmdb_5000_credits.csv"

missing_files = [path.name for path in (MOVIES_FILE, CREDITS_FILE) if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing required file(s): {', '.join(missing_files)}")

movies_raw = pd.read_csv(MOVIES_FILE)
credits_raw = pd.read_csv(CREDITS_FILE)

print(f"Movies:  {movies_raw.shape[0]:,} rows x {movies_raw.shape[1]} columns")
print(f"Credits: {credits_raw.shape[0]:,} rows x {credits_raw.shape[1]} columns")

Movies:  4,803 rows x 20 columns
Credits: 4,803 rows x 4 columns


In [4]:
movies_raw.head(3)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id...",en,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora o...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289}, {""name"": ""Twentieth Century...","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}, {""iso_3166_1"": ""G...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso_639_1"": ""es"", ""name"": ""Espa\u...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""name"": ""drug abuse""}, {""id"": 911...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is hea...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""name"": ""Jerry Bruckheimer Film...","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}]",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 80, ""...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name"": ""based on novel""}, {""id"": 4...",en,Spectre,A cryptic message from Bond’s past sends him on a trail to uncover a siniste...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""name"": ""Danjaq"", ""id"": 10761}, {""...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""}, {""iso_3166_1"": ""US"", ""name""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""}, {""iso_639_1"": ""en"", ""name"": ""...",Released,A Plan No One Escapes,Spectre,6.3,4466


In [5]:
credits_raw.head(3)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""credit_id"": ""5602a8a7c3a368553...","[{""credit_id"": ""52fe48009251416c750aca23"", ""department"": ""Editing"", ""gender""..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Sparrow"", ""credit_id"": ""52fe4232c...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""department"": ""Camera"", ""gender"":..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""credit_id"": ""52fe4d22c3a368484e1...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""department"": ""Sound"", ""gender"": ..."


## Prepare clean movie features

In [6]:
def parse_json_list(value: object) -> list[dict]:
    """Return a parsed list for TMDB JSON-like cells; invalid or empty cells become []."""
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return []
    return parsed if isinstance(parsed, list) else []


def names_from_json(value: object, limit: int | None = None) -> list[str]:
    names = [item.get("name", "").strip() for item in parse_json_list(value) if item.get("name")]
    return names[:limit] if limit else names


def director_from_crew(value: object) -> str:
    for person in parse_json_list(value):
        if person.get("job") == "Director":
            return person.get("name", "").strip()
    return ""


def normalize_token(text: str) -> str:
    """Normalize a phrase into a stable token without losing the human-readable name."""
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def weighted_tokens(row: pd.Series) -> list[str]:
    tokens: list[str] = []

    tokens.extend(f"genre:{normalize_token(name)}" for name in row["genres"])
    tokens.extend(f"keyword:{normalize_token(name)}" for name in row["keywords"])

    # Lead cast and director are stronger taste signals than loose keywords.
    for name in row["cast"]:
        tokens.extend([f"cast:{normalize_token(name)}"] * 2)

    if row["director"]:
        tokens.extend([f"director:{normalize_token(row['director'])}"] * 3)

    return [token for token in tokens if not token.endswith(":")]

In [7]:
credits = credits_raw.copy()
credits["cast"] = credits["cast"].apply(lambda value: names_from_json(value, limit=4))
credits["director"] = credits["crew"].apply(director_from_crew)

movies = movies_raw.merge(
    credits[["movie_id", "cast", "director"]],
    left_on="id",
    right_on="movie_id",
    how="left",
)

movies = movies.assign(
    genres=lambda df: df["genres"].apply(names_from_json),
    keywords=lambda df: df["keywords"].apply(names_from_json),
    cast=lambda df: df["cast"].apply(lambda value: value if isinstance(value, list) else []),
    director=lambda df: df["director"].fillna(""),
)

movies = movies.loc[
    (movies["vote_average"] > 0) & (movies["director"].str.len() > 0),
    ["id", "original_title", "genres", "keywords", "cast", "director", "vote_average", "vote_count", "popularity"],
].reset_index(drop=True)

movies["feature_tokens"] = movies.apply(weighted_tokens, axis=1)
movies["token_counter"] = movies["feature_tokens"].apply(Counter)
movies["token_norm"] = movies["token_counter"].apply(lambda counts: float(np.sqrt(sum(value * value for value in counts.values()))))
movies["new_id"] = np.arange(len(movies))

print(f"Clean movie table: {movies.shape[0]:,} movies")
movies.head(5)

Clean movie table: 4,723 movies


,id,original_title,genres,keywords,cast,director,vote_average,vote_count,popularity,feature_tokens,token_counter,token_norm,new_id
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colony, society, space travel, futu...","[Sam Worthington, Zoe Saldana, Sigourney Weaver, Stephen Lang]",James Cameron,7.2,11800,150.437577,"[genre:action, genre:adventure, genre:fantasy, genre:science_fiction, keywor...","{'genre:action': 1, 'genre:adventure': 1, 'genre:fantasy': 1, 'genre:science...",7.071068,0
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india trading company, love of one's...","[Johnny Depp, Orlando Bloom, Keira Knightley, Stellan Skarsgård]",Gore Verbinski,6.9,4500,139.082615,"[genre:adventure, genre:fantasy, genre:action, keyword:ocean, keyword:drug_a...","{'genre:adventure': 1, 'genre:fantasy': 1, 'genre:action': 1, 'keyword:ocean...",6.633250,1
2,206647,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi6, british secret service, uni...","[Daniel Craig, Christoph Waltz, Léa Seydoux, Ralph Fiennes]",Sam Mendes,6.3,4466,107.376788,"[genre:action, genre:adventure, genre:crime, keyword:spy, keyword:based_on_n...","{'genre:action': 1, 'genre:adventure': 1, 'genre:crime': 1, 'keyword:spy': 1...",5.916080,2
3,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret identity, burglar, hostage dram...","[Christian Bale, Michael Caine, Gary Oldman, Anne Hathaway]",Christopher Nolan,7.6,9106,112.312950,"[genre:action, genre:crime, genre:drama, genre:thriller, keyword:dc_comics, ...","{'genre:action': 1, 'genre:crime': 1, 'genre:drama': 1, 'genre:thriller': 1,...",7.071068,3
4,49529,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel, princess, alien, steampunk, ...","[Taylor Kitsch, Lynn Collins, Samantha Morton, Willem Dafoe]",Andrew Stanton,6.1,2124,43.926995,"[genre:action, genre:adventure, genre:science_fiction, keyword:based_on_nove...","{'genre:action': 1, 'genre:adventure': 1, 'genre:science_fiction': 1, 'keywo...",6.633250,4


## Quick exploration

In [8]:
top_genres = (
    movies.explode("genres")
    .query("genres != ''")
    ["genres"]
    .value_counts()
    .head(10)
    .rename_axis("genre")
    .reset_index(name="movies")
)

top_genres

,genre,movies
0,Drama,2271
1,Comedy,1703
2,Thriller,1265
3,Action,1148
4,Romance,889
5,Adventure,790
6,Crime,691
7,Science Fiction,533
8,Horror,512
9,Family,507


In [9]:
top_directors = (
    movies.loc[movies["director"].ne(""), "director"]
    .value_counts()
    .head(10)
    .rename_axis("director")
    .reset_index(name="movies")
)

top_directors

,director,movies
0,Steven Spielberg,27
1,Woody Allen,21
2,Martin Scorsese,20
3,Clint Eastwood,20
4,Ridley Scott,16
5,Robert Rodriguez,16
6,Spike Lee,16
7,Steven Soderbergh,15
8,Renny Harlin,15
9,Tim Burton,14


## Similarity and recommendation engine

The model uses cosine similarity over weighted feature tokens. It behaves like a KNN recommender: find the nearest movies, then use those neighbors to estimate the selected movie's rating.

In [10]:
def cosine_similarity(counter_a: Counter, norm_a: float, counter_b: Counter, norm_b: float) -> float:
    if norm_a == 0 or norm_b == 0:
        return 0.0

    if len(counter_a) > len(counter_b):
        counter_a, counter_b = counter_b, counter_a

    dot_product = sum(weight * counter_b.get(token, 0) for token, weight in counter_a.items())
    return float(dot_product / (norm_a * norm_b))


def find_movie(title: str) -> pd.Series:
    if not title or not title.strip():
        raise ValueError("Please provide a movie title.")

    query = title.strip().casefold()
    titles = movies["original_title"].str.casefold()

    exact_match = titles.eq(query)
    if exact_match.any():
        return movies.loc[exact_match].iloc[0]

    partial = movies.loc[titles.str.contains(query, regex=False, na=False)].copy()
    if not partial.empty:
        partial["match_start"] = partial["original_title"].str.casefold().str.find(query)
        partial["title_length"] = partial["original_title"].str.len()
        return partial.sort_values(["match_start", "title_length", "vote_count"], ascending=[True, True, False]).iloc[0]

    suggestions = movies.loc[
        titles.str.startswith(query[:1], na=False),
        "original_title",
    ].head(5).tolist()
    hint = f" Try one of these: {suggestions}" if suggestions else ""
    raise LookupError(f"No movie found for '{title}'.{hint}")


def recommend_movies(title: str, k: int = 10) -> pd.DataFrame:
    base_movie = find_movie(title)
    base_counter = base_movie["token_counter"]
    base_norm = base_movie["token_norm"]

    scores = []
    for _, candidate in movies.iterrows():
        if candidate["new_id"] == base_movie["new_id"]:
            continue
        similarity = cosine_similarity(base_counter, base_norm, candidate["token_counter"], candidate["token_norm"])
        scores.append((candidate["new_id"], similarity))

    nearest_ids = [movie_id for movie_id, _ in sorted(scores, key=lambda item: item[1], reverse=True)[:k]]
    nearest = movies.set_index("new_id").loc[nearest_ids].reset_index()
    nearest["similarity"] = [score for _, score in sorted(scores, key=lambda item: item[1], reverse=True)[:k]]

    columns = ["original_title", "genres", "director", "vote_average", "similarity"]
    return nearest[columns]


def predict_score(title: str, k: int = 10) -> tuple[pd.DataFrame, float]:
    selected = find_movie(title)
    recommendations = recommend_movies(selected["original_title"], k=k)
    predicted_rating = float(recommendations["vote_average"].mean())

    print(f"Selected movie: {selected['original_title']}")
    print(f"Actual rating:   {selected['vote_average']:.2f}")
    print(f"Predicted rating from {k} neighbors: {predicted_rating:.2f}")
    return recommendations, predicted_rating

In [11]:
recommend_movies("The Godfather", k=10)

,original_title,genres,director,vote_average,similarity
0,The Godfather: Part III,"[Crime, Drama, Thriller]",Francis Ford Coppola,7.1,0.435897
1,The Godfather: Part II,"[Drama, Crime]",Francis Ford Coppola,8.3,0.435897
2,Apocalypse Now,"[Drama, War]",Francis Ford Coppola,8.0,0.354459
3,The Cotton Club,"[Music, Drama, Crime, Romance]",Francis Ford Coppola,6.6,0.334497
4,The Rainmaker,"[Drama, Crime, Thriller]",Francis Ford Coppola,6.7,0.329541
5,The Outsiders,"[Crime, Drama]",Francis Ford Coppola,6.9,0.306622
6,The Conversation,"[Crime, Drama, Mystery]",Francis Ford Coppola,7.5,0.306622
7,Peggy Sue Got Married,"[Comedy, Drama, Fantasy, Romance]",Francis Ford Coppola,5.9,0.287599
8,New York Stories,"[Comedy, Drama, Romance]",Francis Ford Coppola,6.2,0.268612
9,Twixt,"[Horror, Thriller]",Francis Ford Coppola,5.0,0.247156


In [12]:
recommendations, predicted = predict_score("Donnie Darko", k=10)
recommendations

Selected movie: Donnie Darko
Actual rating:   7.70
Predicted rating from 10 neighbors: 6.46


,original_title,genres,director,vote_average,similarity
0,The Box,"[Thriller, Science Fiction]",Richard Kelly,5.4,0.215821
1,Southland Tales,"[Action, Adventure, Comedy, Drama, Science Fiction, Thriller]",Richard Kelly,5.2,0.189571
2,Ghost,"[Fantasy, Drama, Thriller, Mystery, Romance]",Jerry Zucker,6.9,0.189189
3,Source Code,"[Thriller, Science Fiction, Mystery]",Duncan Jones,7.1,0.160014
4,Into the Wild,"[Adventure, Drama]",Sean Penn,7.8,0.160014
5,Moonlight Mile,"[Romance, Drama]",Brad Silberling,6.5,0.158193
6,Won't Back Down,[Drama],Daniel Barnz,5.8,0.158193
7,Stranger Than Fiction,"[Comedy, Drama, Fantasy, Romance]",Marc Forster,7.1,0.155963
8,Southpaw,"[Action, Drama]",Antoine Fuqua,7.3,0.155342
9,Trust the Man,"[Comedy, Drama, Romance]",Bart Freundlich,5.5,0.152641


In [13]:
for title in ["Godfather", "Notting Hill", "Despicable Me"]:
    print("-" * 72)
    predict_score(title, k=10)
    print()

------------------------------------------------------------------------
Selected movie: The Godfather
Actual rating:   8.40
Predicted rating from 10 neighbors: 6.82

------------------------------------------------------------------------
Selected movie: Notting Hill
Actual rating:   7.00
Predicted rating from 10 neighbors: 6.36

------------------------------------------------------------------------
Selected movie: Despicable Me
Actual rating:   7.10
Predicted rating from 10 neighbors: 6.18



## Smoke checks

In [14]:
assert not movies.empty
assert movies["original_title"].isna().sum() == 0
assert movies["token_norm"].gt(0).any()
assert len(recommend_movies("Avatar", k=5)) == 5

print("All smoke checks passed.")

All smoke checks passed.
